# Feast

In [5]:
!pip cache purge --quiet

In [6]:
!pip install bigtree==1.0.2 \
             cloudpickle==3.1.1 \
             dask==2025.10.0 \
             dill==0.4.0 \
             "feast[singlestore]==0.55.0" \
             gunicorn==23.0.0 \
             mmh3==5.2.0 \
             pyarrow==14.0.1 \
             typeguard==4.4.4 \
             --no-deps --quiet

In [7]:
import IPython

In [8]:
print("Restarting kernel to load new packages...")
print("After restart completes, continue with the next cell.")
IPython.Application.instance().kernel.do_shutdown(restart = True)

Restarting kernel to load new packages...
After restart completes, continue with the next cell.


{'status': 'ok', 'restart': True}

In [5]:
import pandas as pd
import random
import subprocess
import warnings

warnings.filterwarnings("ignore")

from datetime import datetime, timedelta
from feast import FeatureStore
from singlestoredb.management import get_secret

/opt/conda/lib/python3.11/site-packages/feast/permissions/auth_model.py:35: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.float_format", "{:.2f}".format)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [7]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

url = db_connection.url
host = url.host
port = url.port
database = url.database
username = "admin"
password = get_secret("password")

In [8]:
# Create Feature Definitions

os.makedirs("feature_repo/data", exist_ok = True)
os.makedirs("feature_repo/features", exist_ok = True)

feature_store_yaml = f"""
project: ecommerce_features
registry: feature_repo/registry.db
provider: local

online_store:
    type: singlestore
    host: {host}
    port: {port}
    database: {database}
    user: {username}
    password: "{password}"
"""

with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("Feature store configuration created.")

Feature store configuration created.


In [9]:
# Define Features

driver_features_py = """
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String, ValueType
from datetime import timedelta

# Define customer entity
customer = Entity(
    name = "customer_id",
    description = "Customer identifier",
    value_type = ValueType.STRING
)

# Define feature view for customer metrics
customer_features = FeatureView(
    name = "customer_features",
    entities = [customer],
    ttl = timedelta(days = 90),
    schema = [
        Field(name = "total_orders", dtype = Int64),
        Field(name = "total_spent", dtype = Float32),
        Field(name = "avg_order_value", dtype = Float32),
        Field(name = "days_since_last_order", dtype = Int64),
        Field(name = "favorite_category", dtype = String),
    ],
    online = True,
    source = FileSource(
        path = "data/customer_features.parquet",
        timestamp_field = "event_timestamp"
    ),
)
"""

with open("feature_repo/features/driver_features.py", "w") as f:
    f.write(driver_features_py)

print("Feature definitions created.")

Feature definitions created.


In [10]:
# Create Schema and Generate Sample Data

def setup_database():
    """Create tables and populate with sample e-commerce data."""

    # Create customer orders table
    with db_connection.connect() as conn:
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS customer_orders (
            customer_id VARCHAR(50),
            order_id VARCHAR(50),
            order_timestamp DATETIME,
            order_value DECIMAL(10, 2),
            product_category VARCHAR(50),
            quantity INT,
            PRIMARY KEY (order_id),
            KEY idx_customer_ts (customer_id, order_timestamp)
        )
        """))
        conn.execute(text("TRUNCATE TABLE customer_orders"))
        conn.commit()

    # Generate sample data
    customers = [f"CUST_{i:04d}" for i in range(1, 101)]
    categories = ["Electronics", "Clothing", "Home", "Books", "Sports"]

    print("Generating sample orders...")

    orders_data = []
    for i in range(1000):
        customer_id = random.choice(customers)
        order_id = f"ORD_{i:06d}"
        # Orders from last 90 days
        days_ago = random.randint(0, 90)
        order_timestamp = datetime.now() - timedelta(days = days_ago, hours = random.randint(0, 23))
        order_value = round(random.uniform(10, 500), 2)
        product_category = random.choice(categories)
        quantity = random.randint(1, 5)

        orders_data.append({
            "customer_id": customer_id,
            "order_id": order_id,
            "order_timestamp": order_timestamp,
            "order_value": order_value,
            "product_category": product_category,
            "quantity": quantity
        })

    # Bulk insert using Pandas
    orders_df = pd.DataFrame(orders_data)
    orders_df.to_sql(
        "customer_orders",
        con = db_connection,
        if_exists = "append",
        index = False,
        chunksize = 1000
    )

    print("Database setup complete with 1000 sample orders.")

# Run setup
setup_database()

Generating sample orders...
Database setup complete with 1000 sample orders.


In [11]:
# Compute Features from Raw Data

def compute_customer_features():
    """Compute customer features from orders table."""

    query = """
    WITH category_counts AS (
        SELECT
            customer_id,
            product_category,
            COUNT(*) as category_count,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY COUNT(*) DESC) as rn
        FROM customer_orders
        GROUP BY customer_id, product_category
    )
    SELECT
        co.customer_id,
        COUNT(*) as total_orders,
        SUM(co.order_value) as total_spent,
        AVG(co.order_value) as avg_order_value,
        DATEDIFF(NOW(), MAX(co.order_timestamp)) as days_since_last_order,
        MAX(cc.product_category) as favorite_category,
        MAX(co.order_timestamp) as event_timestamp
    FROM customer_orders co
    LEFT JOIN category_counts cc ON co.customer_id = cc.customer_id AND cc.rn = 1
    GROUP BY co.customer_id
    """

    df = pd.read_sql(
        query,
        con = db_connection
    )

    # Save as parquet for Feast
    df.to_parquet("feature_repo/data/customer_features.parquet")

    print(f"Computed features for {len(df)} customers.")
    print("\nSample features:")
    print(df.head(4).T)

    return df

# Compute features
features_df = compute_customer_features()

Computed features for 100 customers.

Sample features:
                                         0                    1                    2                    3
customer_id                      CUST_0078            CUST_0068            CUST_0039            CUST_0052
total_orders                             7                   15                   12                    4
total_spent                        2527.83              3243.41              3787.60               914.91
avg_order_value                     361.12               216.23               315.63               228.73
days_since_last_order                    9                    8                    0                    6
favorite_category                 Clothing                 Home               Sports                Books
event_timestamp        2026-07-01 11:19:46  2026-07-02 20:19:46  2026-07-10 00:19:46  2026-07-04 00:19:46


In [12]:
# Initialize Feast and Apply Feature Definitions

fs = FeatureStore(repo_path = "feature_repo")

# Apply feature definitions (creates registry)
print("Applying feature definitions...")

# !feast -c feature_repo apply
# Alternative apply via Python
result = subprocess.run(
    ["feast", "-c", "feature_repo", "apply"],
    capture_output = True,
    text = True
)

print(result.stdout)
if result.returncode == 0:
    print("Features applied successfully.")
else:
    print("Error:", result.stderr)

INFO:feast.infra.registry.registry:Registry file not found. Creating new registry.


Applying feature definitions...
No project found in the repository. Using project name ecommerce_features defined in feature_store.yaml
Applying changes for project ecommerce_features
Deploying infrastructure for customer_features

Features applied successfully.


In [13]:
# Materialize Features to Online Store

end_date = datetime.now()
start_date = end_date - timedelta(days = 90)

print(f"Materializing features from {start_date} to {end_date}...")

result = subprocess.run(
    [
        "feast", "-c", "feature_repo", "materialize",
        start_date.isoformat(),
        end_date.isoformat()
    ],
    capture_output = True,
    text = True
)

print(result.stdout)

if result.returncode == 0:
    print("Features materialized to online store.")
else:
    print("Error:", result.stderr)

Materializing features from 2026-04-11 19:20:13.728836 to 2026-07-10 19:20:13.728836...
Materializing 1 feature views from 2026-04-11 19:20:13+00:00 to 2026-07-10 19:20:13+00:00 into the singlestore online store.

customer_features:

Features materialized to online store.


In [14]:
# Reinitialize the feature store to pick up the applied features
fs = FeatureStore(repo_path = "feature_repo")

# Get features for customers
test_customers = ["CUST_0001", "CUST_0025", "CUST_0050"]
entity_rows = [{"customer_id": cid} for cid in test_customers]

feature_vector = fs.get_online_features(
    features = [
        "customer_features:total_orders",
        "customer_features:total_spent",
        "customer_features:avg_order_value",
        "customer_features:days_since_last_order",
        "customer_features:favorite_category",
    ],
    entity_rows = entity_rows
)

features_df = feature_vector.to_df()
print(features_df)

  customer_id  total_spent  total_orders  avg_order_value favorite_category  days_since_last_order
0   CUST_0001      4499.61            15           299.97              Home                      8
1   CUST_0025      2984.89            11           271.35              Home                      6
2   CUST_0050      3351.76            12           279.31              Home                      2


In [16]:
# Use features for ML prediction

def predict_customer_churn(customer_id):
    """Use features for real-time prediction."""

    features = fs.get_online_features(
        features = [
            "customer_features:total_orders",
            "customer_features:total_spent",
            "customer_features:days_since_last_order",
        ],
        entity_rows = [{"customer_id": customer_id}]
    ).to_df()

    days_since = features["days_since_last_order"].iloc[0]
    total_orders = features["total_orders"].iloc[0]

    if days_since > 60:
        return {"customer_id": customer_id, "churn_risk": "HIGH", "action": "Send 20% discount"}
    elif days_since > 30:
        return {"customer_id": customer_id, "churn_risk": "MEDIUM", "action": "Send personalized email"}
    else:
        return {"customer_id": customer_id, "churn_risk": "LOW", "action": "Continue engagement"}

# Test predictions
for customer in ["CUST_0001", "CUST_0025", "CUST_0050"]:
    prediction = predict_customer_churn(customer)
    print(f"{prediction['customer_id']}: {prediction['churn_risk']} - {prediction['action']}")

CUST_0001: LOW - Continue engagement
CUST_0025: LOW - Continue engagement
CUST_0050: LOW - Continue engagement


In [17]:
# Verify features were materialized to SingleStore
query = "SELECT feature_name, value, event_ts FROM ecommerce_features_customer_features LIMIT 5"

materialized_features = pd.read_sql(
    query,
    con = db_connection
)

print("Features in SingleStore online store:")
print(materialized_features)

Features in SingleStore online store:
            feature_name                   value            event_ts
0  days_since_last_order                b' \x02' 2026-07-08 10:19:46
1      favorite_category     b'\x12\x08Clothing' 2026-07-07 07:19:46
2      favorite_category  b'\x12\x0bElectronics' 2026-07-06 04:19:46
3  days_since_last_order                b' \x06' 2026-07-04 04:19:46
4      favorite_category         b'\x12\x04Home' 2026-06-29 12:19:46


In [18]:
# Query historical features directly from source data

def get_historical_features_from_orders(customer_id, as_of_date):
    """Get features as they existed at a specific date."""

    query = f"""
    SELECT
        customer_id,
        COUNT(*) as total_orders,
        SUM(order_value) as total_spent,
        AVG(order_value) as avg_order_value
    FROM customer_orders
    WHERE customer_id = '{customer_id}'
    AND order_timestamp <= '{as_of_date}'
    GROUP BY customer_id
    """
    result = pd.read_sql(query, db_connection)
    if result.empty:
        print(f"No historical data found for {customer_id} as of {as_of_date}")
    return result

# Example
past_date = (datetime.now() - timedelta(days = 30)).strftime("%Y-%m-%d")
hist_features = get_historical_features_from_orders("CUST_0001", past_date)
print(f"Features for CUST_0001 as of {past_date}:")
print(hist_features)

Features for CUST_0001 as of 2026-06-10:
  customer_id  total_orders  total_spent  avg_order_value
0   CUST_0001            11      3632.40           330.22
